## Environment set up

In [ ]:
import pandas as pd
from pathlib import Path
import json
from fractions import Fraction

CURR_DIR = Path(".").resolve().parent # Didn't use CWD because it would change if my current directory changed as well

RAW_DATA_DIR = CURR_DIR / "data" / "raw" 

RAW_DATA_PATH = RAW_DATA_DIR / "monsters_raw.json"  
  
with open(RAW_DATA_PATH, "r") as f:
    json_obj = json.load(f) #loads the json file into a python object

raw_df = pd.json_normalize(json_obj)
raw_df.head()


# Phase 2: Preprocessing (Data Cleanup)
We will clean:
* **Armor class**: pull out the numbers that are mixed with parnethesis (the types of armor) and convert these strings to ints

* **Stat mods**: pull out the numbers from the parenthesis and symbol and convert to int

* **HP**: just extract the average, static HP for each monster for simplicity

* **Challenge Rating**: pull out the challenge rating number and leave out the XP, convert to int

* **Size**: extract the size for each monster and put it in their own column, replacing meta

* **Speed**: extract the walking speed only, later may add different types of speed like burrowing or flying

* Drop all the data we don't need for the model, including the missing data we can't use (Nan's). This will essientially be everything but the stat values and their respective modifiers

After cleaning these initial features, then I will:
* Map Encode the **Size** catagories: Use map encoding to convert my size text catagories into a language the ML model can understand


**Armor Class Cleaning**

In [ ]:
#Cleaning the Armor class values (no descriptors, just the number)
ac_pattern = r"^(\d+)" # Captures the number at the start of the data 
ac_cleaned = raw_df["Armor Class"].str.extract(ac_pattern, expand=False).astype(int)
raw_df["Armor Class"] = ac_cleaned

**Modifier cleaning**

In [ ]:
#Cleaning the modifier values (removes the parentheses and converts to int)
mod_pattern = r"\s*\(((?:\+?|\-?)\d+)\)" #Regex pattern to capture the modifier value with its sign
dexterity_cleaned = raw_df["DEX_mod"].str.extract(mod_pattern, expand=False).astype(int)
strength_cleaned = raw_df["STR_mod"].str.extract(mod_pattern, expand=False).astype(int)
constitution_cleaned = raw_df["CON_mod"].str.extract(mod_pattern, expand=False).astype(int)
intelligence_cleaned = raw_df["INT_mod"].str.extract(mod_pattern, expand=False).astype(int)
wisdom_cleaned = raw_df["WIS_mod"].str.extract(mod_pattern, expand=False).astype(int)
charisma_cleaned = raw_df["CHA_mod"].str.extract(mod_pattern, expand=False).astype(int)

raw_df["DEX_mod"] = dexterity_cleaned
raw_df["STR_mod"] = strength_cleaned
raw_df["CON_mod"] = constitution_cleaned
raw_df["INT_mod"] = intelligence_cleaned
raw_df["WIS_mod"] = wisdom_cleaned
raw_df["CHA_mod"] = charisma_cleaned

**HP cleaning**

In [ ]:
#Cleaning the HP values (removes the parentheses and converts to int)
hp_pattern = r"\s*(^\d+)"
hp_cleaned = raw_df["Hit Points"].str.extract(hp_pattern, expand=False).astype(int)
raw_df["Hit Points"] = hp_cleaned

**Challenge Rating cleaning**

In [ ]:
#Cleaning the Challenge Rating values
challenge_pattern = r"\s*(^\d+(?:\/\d+)?)" #Catches fractions or just regular numbers for CR's
challenge_cleaned = raw_df["Challenge"].str.extract(challenge_pattern, expand=False).map(lambda x: Fraction(x)).astype(float) #Converts all the CR values to floats, including the fractions
raw_df["Challenge"] = challenge_cleaned

simplifying the **Meta** catagory to **Size** only

In [ ]:
size_pattern = r"\s*^(\w+)"
size = raw_df["meta"].str.extract(size_pattern, expand =False)
raw_df["Size"] = size
raw_df.drop(columns=["meta"], inplace=True)

Simplifying **Speed** catagory to **Walking speed** only

In [ ]:
speed_pattern = r"\s*^(\d+)"
walk_speed = raw_df["Speed"].str.extract(speed_pattern, expand =False)
raw_df["Walking Speed (ft)"] = walk_speed
raw_df.drop(columns=["Speed"], inplace=True)

Dropping all the values we don't need (everything else)

In [ ]:
#Did it in multiple lines instead of just one for readibility purposes
raw_df.drop(columns=["STR", "DEX", "CON", "INT", "WIS", "CHA"], inplace=True) # Drop the raw stat scores
raw_df.drop(columns=["Saving Throws", "Skills", "Senses", "Languages", "Traits", "Actions", "Legendary Actions", "img_url"], inplace=True)
raw_df.drop(columns=["Damage Immunities", "Condition Immunities","Damage Resistances" ,"Damage Vulnerabilities", "Reactions"], inplace=True)


Map encoding my **Size** catagories (Tiny, Small, Medium, Large, Huge, Gargantuan)

In [ ]:

size_order = ['Tiny', 'Small', 'Medium', 'Large', 'Huge', 'Gargantuan'] 
encoded_sizes = pd.Categorical(values=raw_df["Size"], categories=size_order, ordered=True).codes

#Encoded values: Tiny = 0, Small = 1, Medium = 2, Large = 3, Huge = 4, Gargantuan = 5
raw_df["Encoded Sizes"] = encoded_sizes 

raw_df.head()


## Saving it to the User's device

In [ ]:
PROCESSED_DATA_DIR = CURR_DIR / "data" / "processed" # Simalar to what we did earlier
PROCESSED_DATA_PATH = PROCESSED_DATA_DIR / "monsters_processed.json"  

if not PROCESSED_DATA_PATH.exists(): # checks if folder archetechure exists on user device. If not creates it
    print("File Path not found! Creating it now...")
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True) # Makes the directory on the user's local machine. If its already there or needs parents then it handles it
    with open(PROCESSED_DATA_PATH, "w") as f:
        f.write(raw_df.to_json()) #caches the data set onto the user's local machine
else: 
    print("File path validated! The cleaned dataset is on your computer!")